# Docling PDF Header and Footer Extraction

Notebook to convert a PDF with Docling, inspect the furniture tree, extract elements labeled as page headers or footers, and save the results as JSON.

**Note:** The furniture tree contains elements Docling has already identified; it cannot recover headers or footers the parser missed. Object structure can vary by Docling version.


In [1]:
!python --version

Python 3.14.8


In [1]:
%pip install -q docling docling-core

^C
Note: you may need to restart the kernel to use updated packages.


## 1. Set the input PDF

Place the PDF next to this notebook or update `PDF_PATH` to its location.

In [13]:
from pathlib import Path
import json

PDF_PATH = Path("79MSE010710_F.pdf")
OUTPUT_PATH = Path("headers_footers.json")

print("Input PDF exists:", PDF_PATH.is_file())
print("Expected path:", PDF_PATH.resolve())

Input PDF exists: True
Expected path: /Users/suyashutekar/Projects/pdflens_replacement/79MSE010710_F.pdf


## 2. Convert the PDF with Docling

In [ ]:
from docling.document_converter import DocumentConverter

/Users/suyashutekar/Projects/pdflens_replacement/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 14:44:45.474000 1476 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 14989.53it/s]
/Users/suyashutekar/Projects/pdflens_replacement/.venv/lib/python3.14/site-packages/rapidocr/inference_engine/pytorch/networks/heads/multiheadAttention.py:14: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  \text{MultiHead}(Q, K, V) = \text{Concat}(head_1,\dots,head_h)W^O
/Users/suyashutekar/Projects/pdflens_replacement/.venv/lib/python3.14/site-packages/torch/nn/modules/conv.py:560: UserWarning: 

Conversion status: ConversionStatus.SUCCESS
Document type: DoclingDocument


In [14]:
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"Update PDF_PATH to an existing PDF: {PDF_PATH}")

converter = DocumentConverter()
conversion_result = converter.convert(str(PDF_PATH))
doc = conversion_result.document

print("Conversion status:", conversion_result.status)
print("Document type:", type(doc).__name__)

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 14237.41it/s]


Conversion status: ConversionStatus.SUCCESS
Document type: DoclingDocument


## 3. Inspect the furniture tree and text labels

In [15]:
furniture = getattr(doc, "furniture", None)
print("Furniture attribute available:", hasattr(doc, "furniture"))
print("Furniture object type:", type(furniture).__name__ if furniture is not None else None)
print(str(furniture)[:2500] if furniture is not None else "No furniture object exposed.")

Furniture attribute available: True
Furniture object type: GroupItem
self_ref='#/furniture' parent=None children=[] content_layer=<ContentLayer.FURNITURE: 'furniture'> meta=None name='_root_' label=<GroupLabel.UNSPECIFIED: 'unspecified'>


/var/folders/_9/2mhpzqrx5b14gz4g_fq712zh0000gn/T/ipykernel_1476/1240090298.py:1: DeprecationWarning: deprecated
  furniture = getattr(doc, "furniture", None)
/var/folders/_9/2mhpzqrx5b14gz4g_fq712zh0000gn/T/ipykernel_1476/1240090298.py:2: DeprecationWarning: deprecated
  print("Furniture attribute available:", hasattr(doc, "furniture"))


In [16]:
def normalize_label(item):
    label = getattr(item, "label", None)
    if label is None:
        return ""
    return str(getattr(label, "value", label)).lower()

texts = getattr(doc, "texts", []) or []
label_counts = {}
for item in texts:
    label = normalize_label(item)
    label_counts[label] = label_counts.get(label, 0) + 1

print("Label counts:", label_counts)

for item in texts:
    label = normalize_label(item)
    if any(word in label for word in ("header", "footer", "title")):
        print({"label": label, "text": getattr(item, "text", None),
               "self_ref": str(getattr(item, "self_ref", None))})

Label counts: {'text': 143, 'section_header': 42, 'list_item': 24, 'page_footer': 41, 'page_header': 40, 'caption': 1}
{'label': 'section_header', 'text': 'The Chairman/Managing Director/ Chief Executive Officer', 'self_ref': '#/texts/3'}
{'label': 'section_header', 'text': 'Master Circular - Lending to Micro, Small & Medium Enterprises (MSME) Sector', 'self_ref': '#/texts/6'}
{'label': 'page_footer', 'text': 'Tel No: +91 22 2261 0261| Fax No: +91 22 2262 1011/2265 8273/2265 8276| Email: cgmicrpcd@rbi.org.in', 'self_ref': '#/texts/12'}
{'label': 'section_header', 'text': 'R RE ES SE ER RV VE E B BA AN NK K O OF F I IN ND DI IA A', 'self_ref': '#/texts/13'}
{'label': 'page_footer', 'text': 'Rural Planning & Credit Department Central Office, 10 th Floor, Central Office Building, Post Box no. 10014 Mumbai - 400 001', 'self_ref': '#/texts/15'}
{'label': 'page_header', 'text': 'Master Circular Lending to Micro, Small & Medium Enterprises Sector', 'self_ref': '#/texts/16'}
{'label': 'section

## 4. Extract headers and footers

The notebook first traverses `doc.furniture` if exposed, then checks `doc.texts` for explicit `PAGE_HEADER` and `PAGE_FOOTER` labels. It does not treat section headings as page headers.

In [17]:
def get_bbox(prov):
    bbox = getattr(prov, "bbox", None)
    if bbox is None:
        return None
    return {
        "left": getattr(bbox, "l", None),
        "top": getattr(bbox, "t", None),
        "right": getattr(bbox, "r", None),
        "bottom": getattr(bbox, "b", None),
        "coordinate_origin": str(getattr(bbox, "coord_origin", "")),
    }

def extract_item(item, category):
    text = getattr(item, "text", None)
    if not text or not text.strip():
        return None
    provenance = getattr(item, "prov", []) or []
    locations = [
        {"page_number": getattr(p, "page_no", None), "bbox": get_bbox(p)}
        for p in provenance
    ]
    return {
        "type": category,
        "text": text.strip(),
        "docling_label": normalize_label(item),
        "item_ref": str(getattr(item, "self_ref", None)),
        "locations": locations,
    }

def resolve_ref(ref, document):
    try:
        return ref.resolve(doc=document)
    except (AttributeError, TypeError, ValueError):
        return None

def walk_tree(node, document, results, visited):
    if node is None:
        return
    item_ref = getattr(node, "self_ref", None)
    if item_ref is not None:
        ref_key = str(item_ref)
        if ref_key in visited:
            return
        visited.add(ref_key)

    label = normalize_label(node)
    if label in {"page_header", "page_footer"}:
        record = extract_item(node, label)
        if record:
            results.append(record)

    for child_ref in getattr(node, "children", []) or []:
        child = resolve_ref(child_ref, document)
        if child is not None:
            walk_tree(child, document, results, visited)

def extract_headers_footers(document):
    results = []
    visited = set()

    # Primary path: furniture tree, if exposed by this Docling version.
    furniture_node = getattr(document, "furniture", None)
    if furniture_node is not None:
        walk_tree(furniture_node, document, results, visited)

    # Fallback path: explicitly labeled text items.
    for item in getattr(document, "texts", []) or []:
        label = normalize_label(item)
        if label in {"page_header", "page_footer"}:
            record = extract_item(item, label)
            if record:
                results.append(record)

    # Deduplicate items found through both paths.
    unique, seen = [], set()
    for record in results:
        key = (record["item_ref"], record["type"], record["text"],
               json.dumps(record["locations"], sort_keys=True))
        if key not in seen:
            seen.add(key)
            unique.append(record)
    return unique

records = extract_headers_footers(doc)
headers = [r for r in records if r["type"] == "page_header"]
footers = [r for r in records if r["type"] == "page_footer"]

print("Headers found:", len(headers))
print("Footers found:", len(footers))
records[:10]

Headers found: 40
Footers found: 41


/var/folders/_9/2mhpzqrx5b14gz4g_fq712zh0000gn/T/ipykernel_1476/546710677.py:62: DeprecationWarning: deprecated
  furniture_node = getattr(document, "furniture", None)


[{'type': 'page_footer',
  'text': 'Tel No: +91 22 2261 0261| Fax No: +91 22 2262 1011/2265 8273/2265 8276| Email: cgmicrpcd@rbi.org.in',
  'docling_label': 'page_footer',
  'item_ref': '#/texts/12',
  'locations': [{'page_number': 1,
    'bbox': {'left': 97.19688000000005,
     'top': 54.96018000000004,
     'right': 498.0857459999999,
     'bottom': 47.80260926004223,
     'coordinate_origin': 'CoordOrigin.BOTTOMLEFT'}}]},
 {'type': 'page_footer',
  'text': 'Rural Planning & Credit Department Central Office, 10 th Floor, Central Office Building, Post Box no. 10014 Mumbai - 400 001',
  'docling_label': 'page_footer',
  'item_ref': '#/texts/15',
  'locations': [{'page_number': 1,
    'bbox': {'left': 93.9,
     'top': 74.29571999999996,
     'right': 501.2721600000003,
     'bottom': 57.09930926004233,
     'coordinate_origin': 'CoordOrigin.BOTTOMLEFT'}}]},
 {'type': 'page_header',
  'text': 'Master Circular Lending to Micro, Small & Medium Enterprises Sector',
  'docling_label': 'page

## 5. Save results to JSON

In [18]:
output = {
    "source_pdf": PDF_PATH.name,
    "conversion_status": str(conversion_result.status),
    "header_footer_count": len(records),
    "headers": headers,
    "footers": footers,
}

OUTPUT_PATH.write_text(
    json.dumps(output, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Saved:", OUTPUT_PATH.resolve())

Saved: /Users/suyashutekar/Projects/pdflens_replacement/headers_footers.json


## 6. Troubleshooting

- **No header/footer labels found:** inspect the raw Docling export and the labels available in your installed version.
- **Section headings appear as headers:** correct the adapter mapping; section headings and repeating page headers are different element types.
- **Running header/footer is absent:** add a separate PyMuPDF fallback using page coordinates and repeated-text patterns across pages.
- **Scanned PDFs:** validate OCR quality first.
- **Production testing:** test changing headers, page numbers, multi-column layouts, and scanned PDFs.

The script does not infer missing furniture based on position alone; that should be a separately tested fallback.
